# Cvičení 2: XGBoost – Regresní model pro predikci cen diamantů

> **Cíl cvičení:** Vytvořit regresní model založený na algoritmu **XGBoost** (`XGBRegressor`) pro predikci ceny diamantů (`price`).
> Rozdělit data v poměru 70:30 (`random_state=42`), definovat hyperparametrickou mřížku pro `RandomizedSearchCV`,
> optimalizovat model na metriku `mean_absolute_error` a vyhodnotit průměrnou absolutní chybu (MAE) na testovací sadě.

---

## 1. Import knihoven a načtení dat

Použijeme dataset `diamonds_preprocessed.csv`, který byl připraven pro regresní cvičení v kurzu (kategoriální proměnné jsou již kódovány).
Importujeme třídy z `pandas`, `sklearn.model_selection`, `sklearn.metrics` a knihovny `xgboost`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

# Načtení dat
data_path = "data/diamonds_preprocessed.csv"
df = pd.read_csv(data_path)

print(f"Rozměry datasetu: {df.shape}")
df.head()

## 2. Příprava dat a rozdělení na Train / Test (70:30)

Oddělíme cílovou proměnnou `price` od prediktorů a rozdělíme dataset v poměru 70:30 s fixním seedem `random_state=42`.

In [ ]:
X = df.drop("price", axis=1)
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print(f"Počet trénovacích vzorků: {len(X_train):,}")
print(f"Počet testovacích vzorků:   {len(X_test):,}")

## 3. Vytvoření instance `xgb_regressor` a definice hyperparametrů

Zadání požaduje vytvořit instanci s parametry `random_state=42` a `n_jobs=-1`.
Slovník `params` obsahuje klíče:
- `max_depth`: [3, 6, 9]
- `min_samples_leaf`: [1, 2, 4]  *(Pozn.: V XGBoostu jde o parametr zadaný v zadání kurzu; interně v XGBoost odpovídá parametru min_child_weight)*
- `n_estimators`: [50, 100, 150]

In [ ]:
# Instance XGBoost regresoru
xgb_regressor = xgb.XGBRegressor(random_state=42, n_jobs=-1)

# Mřížka hyperparametrů dle zadání
params = {
    "max_depth": [3, 6, 9],
    "min_samples_leaf": [1, 2, 4],
    "n_estimators": [50, 100, 150]
}

print("Parametry pro RandomizedSearchCV:", params)

## 4. Ladění modelu pomocí `RandomizedSearchCV`

V Scikit-learn je optimalizační metrika pro Mean Absolute Error definována jako `neg_mean_absolute_error`.
Natrénujeme mřížku s křížovou validací.

In [ ]:
random_search = RandomizedSearchCV(
    estimator=xgb_regressor,
    param_distributions=params,
    scoring="neg_mean_absolute_error",
    n_iter=8,
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

print(f"Nejlepší hyperparametry: {random_search.best_params_}")
print(f"Nejlepší CV MAE (trénovací křížová validace): {-random_search.best_score_:.2f} USD")

## 5. Predikce a vyhodnocení na testovací sadě

Pomocí nejlepšího modelu provedeme predikci na `X_test` a spočteme testovací metriky (MAE, RMSE, R²).

In [ ]:
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== VÝSLEDKY NA TESTOVACÍ SADĚ (XGBoost) ===")
print(f"Testovací MAE:  {mae:.2f} USD")
print(f"Testovací RMSE: {rmse:.2f} USD")
print(f"Testovací R²:   {r2:.4f} ({r2*100:.2f} % vysvětleného rozptylu)")

## 6. Srovnání modelů a důležitost příznaků (Feature Importances)

Podíváme se, jaké příznaky považuje nejlepší XGBoost model za nejdůležitější pro predikci ceny diamantů.

In [ ]:
importance_df = pd.DataFrame({
    "Příznak": X.columns,
    "Důležitost (Gain)": best_model.feature_importances_
}).sort_values(by="Důležitost (Gain)", ascending=False)

importance_df

## 7. Expertní rozšíření: Nativní ladění XGBoost (SOTA 2026)

Všimněte si, že XGBoost vypsal varování: `Parameters: { "min_samples_leaf" } are not used.`
V XGBoostu se váha listu a minimální počet instancí řídí parametrem `min_child_weight`.
Pokud přidáme subsampling sloupců `colsample_bytree` a rychlost učení `learning_rate`, dosáhneme ještě lepší chyby:

In [ ]:
sota_params = {
    "max_depth": [6, 8, 10],
    "n_estimators": [100, 150, 200],
    "learning_rate": [0.05, 0.1, 0.15],
    "min_child_weight": [1, 3, 5],
    "colsample_bytree": [0.8, 1.0]
}

sota_rs = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=sota_params,
    scoring="neg_mean_absolute_error",
    n_iter=8,
    cv=3,
    random_state=42,
    n_jobs=-1
)
sota_rs.fit(X_train, y_train)

y_pred_sota = sota_rs.best_estimator_.predict(X_test)
sota_mae = mean_absolute_error(y_test, y_pred_sota)
print(f"SOTA XGBoost Test MAE: {sota_mae:.2f} USD (zlepšení o {mae - sota_mae:.2f} USD!)")